# Bellman Capital — Runner para Google Colab

Notebook autocontenido. `Entorno de ejecución` → `Ejecutar todas`.
Recomendado activar GPU: `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → `T4 GPU`.

**Agente final:** menú long-only · rebalanceo semanal · reward drawdown-penalized · Double DQN.

## 1. Clonar el repositorio (incluye los datos)

In [ ]:
%cd /content
!rm -rf bellmancapital
!git clone https://github.com/emiliomunozai/bellmancapital.git
%cd /content/bellmancapital
!ls -lh data/raw/*.parquet

## 2. Instalar dependencias

In [ ]:
!pip install -q "gymnasium>=0.29" pytest
import torch; print("torch:", torch.__version__, "| GPU:", torch.cuda.is_available())

## 3. Escribir `agent.py` (la entrega)

In [ ]:
%%writefile agent.py
# =============================================================================
# BELLMAN CAPITAL — agent.py
# Único archivo de entrega. Define TradingEnv y Agent. No modifica nada en src/.
#
# Resumen de diseño (ver informe del Punto 1 para la justificación completa):
#   - Intervalo objetivo: 1h. Lookback de 24 (un día de historia).
#   - State: log-returns (24 x 3 activos) + régimen (vol, momentum) + pesos
#            actuales + indicador de rebalanceo. Todo lookahead-safe.
#   - Action space: 10 acciones discretas long-only (9 carteras + HOLD).
#            El orden respeta los baselines en src/baselines.py.
#   - Rebalanceo periódico: el agente propone cada paso, el entorno aplica el
#            cambio cada 168 pasos (semanal) -> preserva el edge bajo costos.
#   - Reward: log-return del portafolio (neto de costos) menos penalización de
#            turnover. Respeta los signos exigidos por los tests.
#   - Algoritmo: Double DQN con target network y replay buffer.
# =============================================================================

import random
from collections import deque

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from gymnasium import spaces

from src.env import BaseTradingEnv
from src.base import BaseAgent


# ── Action space ────────────────────────────────────────────────────────────
# Pesos por accion: [asset_0, asset_1, asset_2, cash]. Cada fila suma 1,
# cash >= 0, pesos riesgosos en [0, 1]. El ORDEN respeta los baselines:
#   accion 0 = HoldCash, accion 1 = HoldAsset0, accion 4 = EqualWeight.
# Menu LONG-ONLY: el EDA del Punto 1 mostro drift alcista estructural fuerte;
# las posiciones cortas resultaron -EV y amplificaban el distribution shift
# (un agente entrenado en periodos bear shorteaba en periodos bull). Se exploro
# con shorts (acciones [-0.5,..,1.5]) y empeoraban el desempeno, asi que el menu
# final es long-only + cash.
# La ultima accion (HOLD) mantiene los pesos actuales (turnover 0), dando al
# agente una forma explicita de no pagar costos. Su fila es nominal (all-cash);
# _weights_from_action la intercepta.
_ACTION_WEIGHTS = np.array(
    [
        [0.0,      0.0,      0.0,      1.0],   # 0  todo cash
        [1.0,      0.0,      0.0,      0.0],   # 1  todo asset_0
        [0.0,      1.0,      0.0,      0.0],   # 2  todo asset_1
        [0.0,      0.0,      1.0,      0.0],   # 3  todo asset_2
        [1/3,      1/3,      1/3,      0.0],   # 4  equal weight riesgoso
        [0.5,      0.0,      0.0,      0.5],   # 5  half-bet asset_0 + cash
        [0.0,      0.5,      0.0,      0.5],   # 6  half-bet asset_1 + cash
        [0.0,      0.0,      0.5,      0.5],   # 7  half-bet asset_2 + cash
        [1/6,      1/6,      1/6,      0.5],   # 8  half equal-weight + cash
        [0.0,      0.0,      0.0,      1.0],   # 9  HOLD (mantiene pesos actuales)
    ],
    dtype=np.float32,
)
N_ACTIONS = len(_ACTION_WEIGHTS)
_HOLD_ACTION = N_ACTIONS - 1   # indice de la accion HOLD


# ── Environment ───────────────────────────────────────────────────────────────

class TradingEnv(BaseTradingEnv):

    # cuantos pasos de historia ve el agente (un dia a 1h)
    LOOKBACK = 24

    # penalizacion extra por turnover en el reward (amplifica la senal de costo
    # para que supere el ruido del retorno horario). 0.0 = solo log-return.
    TURNOVER_PENALTY = 5e-4

    # frecuencia de rebalanceo: el agente propone cada paso, pero el entorno solo
    # aplica el cambio cada REBALANCE_EVERY pasos. A 1h, 168 = una semana.
    # El EDA mostro que el agente tiene senal direccional (gana a 0 bps) pero su
    # edge moria por costos al rebalancear a diario; el rebalanceo semanal reduce
    # el turnover ~8x y preserva el desempeno bajo costos realistas.
    REBALANCE_EVERY = 168

    # formulacion del reward (ver Seccion 4 del informe):
    #   "log"      -> log-return puro
    #   "turnover" -> log-return - lambda*turnover
    #   "drawdown" -> log-return - mu*drawdown_actual   (FINAL; el que se entrena)
    # Con rebalanceo semanal el turnover ya esta controlado estructuralmente, asi
    # que penalizar el drawdown (alineado con el Sortino, la metrica objetivo)
    # rinde mejor en todos los regimenes probados, incluido el bear de 2022.
    REWARD_MODE = "drawdown"
    DRAWDOWN_PENALTY = 0.10

    def __init__(self, prices, transaction_cost_bps=10.0, initial_cash=10_000.0):
        super().__init__(prices, transaction_cost_bps, initial_cash)

        self._lookback = self.LOOKBACK
        self._initial_cash = float(initial_cash)

        # turnover del ultimo rebalanceo (0 hasta el primer step real; mantener
        # en 0 hace que los tests aislados de _reward devuelvan log-return puro).
        self._last_turnover = 0.0
        # pico del valor del portafolio, para el reward drawdown-penalized
        self._peak_value = float(initial_cash)

        self.action_space = spaces.Discrete(N_ACTIONS)

        # log-returns: LOOKBACK pasos x 3 activos riesgosos  -> 3 * LOOKBACK
        # regimen:     vol (3) + momentum (3)                 -> 6
        # posicion:    pesos actuales del portafolio          -> 4
        # fase:        indicador de "toca rebalancear"         -> 1
        obs_dim = 3 * self._lookback + 6 + 4 + 1
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(obs_dim,), dtype=np.float32
        )

        # precios de los 3 activos riesgosos (sin la columna cash, que es constante)
        self._risky = self.prices[:, :3]

    def _is_rebalance_step(self) -> bool:
        # self._t aun no se incrementa cuando se evalua dentro de step()
        return (self._t % self.REBALANCE_EVERY) == 0

    # -- reset: reinicia el estado propio al empezar cada episodio ------------
    def reset(self, *args, **kwargs):
        obs, info = super().reset(*args, **kwargs)
        self._peak_value = float(getattr(self, "_value", self._initial_cash))
        self._last_turnover = 0.0
        return obs, info

    # -- observacion ----------------------------------------------------------
    def _obs(self) -> np.ndarray:
        # en el paso terminal BaseTradingEnv llama _obs con self._t == len(prices);
        # acotamos al ultimo indice valido (ese obs no se usa para decidir).
        t = min(self._t, len(self._risky) - 1)
        L = self._lookback

        # --- bloque 1: ultimos L log-returns de cada activo riesgoso ---------
        # ventana de precios [t-L, ..., t]  -> L returns, lookahead-safe
        window = self._risky[t - L:t + 1]                        # (L+1, 3)
        rets = np.log((window[1:] + 1e-8) / (window[:-1] + 1e-8))  # (L, 3)
        rets = rets * 100.0                                      # llevar a orden ~1
        ret_block = rets.reshape(-1)                             # (L*3,) tiempo-mayor

        # --- bloque 2: features de regimen (volatilidad y momentum) ----------
        vol = rets.std(axis=0)                                   # (3,)
        m = min(20, t)
        mom = np.log((self._risky[t] + 1e-8) / (self._risky[t - m] + 1e-8))  # (3,)

        # --- bloque 3: pesos actuales del portafolio -------------------------
        weights = self._weights.astype(np.float32)               # (4,)

        # --- bloque 4: indicador de fase (1 si la proxima accion rebalancea) --
        phase = np.array([1.0 if (t % self.REBALANCE_EVERY) == 0 else 0.0],
                         dtype=np.float32)

        obs = np.concatenate([ret_block, vol, mom, weights, phase]).astype(np.float32)
        # robustez numerica: sin NaN/Inf y acotado para gradientes estables
        obs = np.nan_to_num(obs, nan=0.0, posinf=10.0, neginf=-10.0)
        obs = np.clip(obs, -10.0, 10.0).astype(np.float32)
        return obs

    # -- mapeo accion -> pesos ------------------------------------------------
    def _weights_from_action(self, action: int) -> np.ndarray:
        action = int(action)
        if not self._is_rebalance_step() or action == _HOLD_ACTION:
            # fuera de la ventana de rebalanceo, o accion HOLD:
            # mantener la cartera actual -> turnover 0, sin costo.
            w = self._weights.astype(np.float32).copy()
        else:
            w = _ACTION_WEIGHTS[action].copy()
        # self._weights todavia son los pesos previos en este punto del step,
        # asi que capturamos el turnover del rebalanceo para usarlo en _reward.
        self._last_turnover = float(np.abs(w - self._weights).sum())
        return w

    # -- reward ---------------------------------------------------------------
    def _reward(self, prev_value: float, curr_value: float) -> float:
        # En todos los modos: > 0 si el valor crece, < 0 si cae, y exactamente 0
        # si no cambia (con turnover 0 y sin drawdown), respetando los tests.
        prev = max(float(prev_value), 1e-8)
        curr = max(float(curr_value), 1e-8)
        log_ret = float(np.log(curr / prev))

        if self.REWARD_MODE == "log":
            return log_ret

        if self.REWARD_MODE == "drawdown":
            self._peak_value = max(self._peak_value, curr)
            dd = (self._peak_value - curr) / self._peak_value   # >= 0
            return log_ret - self.DRAWDOWN_PENALTY * dd

        # "turnover" (final): el valor ya viene neto de costos; la penalizacion
        # explicita amplifica la senal de costo sobre el ruido del retorno.
        return log_ret - self.TURNOVER_PENALTY * self._last_turnover


# ── DQN network ───────────────────────────────────────────────────────────────

class _QNetwork(nn.Module):
    def __init__(self, obs_dim: int, n_actions: int, hidden=(128, 128)):
        super().__init__()
        layers, d = [], obs_dim
        for h in hidden:
            layers += [nn.Linear(d, h), nn.ReLU()]
            d = h
        layers += [nn.Linear(d, n_actions)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


# ── Replay buffer ───────────────────────────────────────────────────────────

class _ReplayBuffer:
    def __init__(self, capacity: int):
        self.buf = deque(maxlen=capacity)

    def push(self, s, a, r, s2, done):
        self.buf.append((s, a, r, s2, done))

    def sample(self, batch_size: int):
        batch = random.sample(self.buf, batch_size)
        s, a, r, s2, d = zip(*batch)
        return (
            np.array(s, dtype=np.float32),
            np.array(a, dtype=np.int64),
            np.array(r, dtype=np.float32),
            np.array(s2, dtype=np.float32),
            np.array(d, dtype=np.float32),
        )

    def __len__(self):
        return len(self.buf)


# ── Agent ─────────────────────────────────────────────────────────────────────

class Agent(BaseAgent):

    def __init__(self, obs_dim: int, n_actions: int):
        super().__init__(obs_dim, n_actions)

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # hiperparametros (ver configs/default.yaml)
        self.gamma = 0.99
        self.lr = 1e-4
        self.batch_size = 64
        self.target_update_freq = 1000
        self.learning_starts = 200
        self.epsilon_start = 1.0
        self.epsilon_end = 0.05
        self.epsilon_decay_steps = 50_000

        # exploracion
        self.epsilon = self.epsilon_start
        self._steps_done = 0

        # redes
        self.q = _QNetwork(obs_dim, n_actions).to(self.device)
        self.target = _QNetwork(obs_dim, n_actions).to(self.device)
        self.target.load_state_dict(self.q.state_dict())
        self.target.eval()

        self.optimizer = torch.optim.Adam(self.q.parameters(), lr=self.lr)
        self.buffer = _ReplayBuffer(100_000)

    # -- schedule de epsilon --------------------------------------------------
    def _update_epsilon(self):
        frac = min(1.0, self._steps_done / self.epsilon_decay_steps)
        self.epsilon = self.epsilon_start + frac * (self.epsilon_end - self.epsilon_start)

    # -- politica epsilon-greedy (entrenamiento) ------------------------------
    def _epsilon_greedy(self, obs: np.ndarray) -> int:
        if random.random() < self.epsilon:
            return random.randrange(self.n_actions)
        return self.act(obs)

    # -- paso de aprendizaje (Double DQN) -------------------------------------
    def _learn(self):
        s, a, r, s2, d = self.buffer.sample(self.batch_size)
        s  = torch.as_tensor(s,  device=self.device)
        a  = torch.as_tensor(a,  device=self.device).unsqueeze(1)
        r  = torch.as_tensor(r,  device=self.device).unsqueeze(1)
        s2 = torch.as_tensor(s2, device=self.device)
        d  = torch.as_tensor(d,  device=self.device).unsqueeze(1)

        q_sa = self.q(s).gather(1, a)
        with torch.no_grad():
            # Double DQN: la red online elige la accion, la target la evalua
            next_actions = self.q(s2).argmax(dim=1, keepdim=True)
            q_next = self.target(s2).gather(1, next_actions)
            target = r + self.gamma * q_next * (1.0 - d)

        loss = F.smooth_l1_loss(q_sa, target)
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.q.parameters(), 10.0)
        self.optimizer.step()

    # -- bucle de entrenamiento -----------------------------------------------
    def train(self, env, n_steps: int = 200_000) -> None:
        self.q.train()
        obs, _ = env.reset()
        for _ in range(n_steps):
            self._update_epsilon()
            action = self._epsilon_greedy(obs)
            next_obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            self.buffer.push(obs, action, reward, next_obs, float(done))

            if len(self.buffer) >= max(self.batch_size, self.learning_starts):
                self._learn()

            self._steps_done += 1
            if self._steps_done % self.target_update_freq == 0:
                self.target.load_state_dict(self.q.state_dict())

            obs = next_obs if not done else env.reset()[0]
        self.q.eval()

    # -- politica greedy (evaluacion, deterministica) -------------------------
    def act(self, obs: np.ndarray) -> int:
        with torch.no_grad():
            x = torch.as_tensor(np.asarray(obs, dtype=np.float32),
                                device=self.device).unsqueeze(0)
            q = self.q(x)
            return int(q.argmax(dim=1).item())


## 4. Verificar los 54 tests (lo crítico para la entrega)

In [ ]:
!python -m pytest tests/test_submission.py -v

## 5. Escribir los scripts de evaluación

In [ ]:
%%writefile run_experiment.py
"""
BELLMAN CAPITAL — evaluación walk-forward del agente contra los baselines.

Produce:
  - tabla de métricas por ventana (cumret, sortino, sharpe, maxdd, fees) a 0/10/25 bps
  - equity curves agente vs baselines (con spread de semillas en el último split)
  - allocation plot del agente en el tiempo
  - todo guardado en results/

Uso:
    python run_experiment.py --interval 1h --steps 200000 --seeds 3
    python run_experiment.py --quick          # corrida corta de prueba

No modifica src/ ni agent.py. Importa de ambos.
"""
import os
import argparse
import json
import random

import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from src.data import load_prices, split
from src.metrics import compute_metrics
from src.baselines import RandomPolicy, HoldCash, HoldAsset0, EqualWeight, SMA
import agent as A
from agent import TradingEnv, Agent, N_ACTIONS

# ventanas walk-forward (de configs/default.yaml — NO alterar)
SPLITS = [
    ("2019-12-31", "2020-12-31"),
    ("2020-12-31", "2021-12-31"),
    ("2021-12-31", "2022-12-31"),
    ("2022-12-31", "2023-12-31"),
    ("2023-12-31", "2025-12-31"),
]
FREQ = {"1h": 24 * 365, "30m": 48 * 365, "15m": 96 * 365}


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)


def run_policy(policy, prices, tc_bps, sma_lookback=None):
    """Corre una política sobre un período y devuelve (values, turnovers)."""
    env = TradingEnv(prices, transaction_cost_bps=tc_bps)
    obs, _ = env.reset()
    done = False
    vals, turns = [], []
    while not done:
        if sma_lookback is not None:                  # SMA lee retornos crudos
            a = policy.act(obs[:sma_lookback * 3])
        else:
            a = policy.act(obs)
        obs, _, term, trunc, info = env.step(a)
        done = term or trunc
        vals.append(info["portfolio_value"])
        turns.append(info["turnover"])
    return np.array(vals), np.array(turns)


def metrics_row(name, vals, turns, tc_bps, freq, initial=10_000.0):
    m = compute_metrics(vals, freq=freq)
    # fees aproximados: turnover * tc * valor (acumulado)
    fees = float(np.sum(turns) * (tc_bps / 10_000) * initial)
    return {
        "policy": name, "tc_bps": tc_bps,
        "cum_ret": m["cum_ret"], "sortino": m["sortino"],
        "sharpe": m["sharpe"], "max_dd": m["max_dd"],
        "total_turnover": float(np.sum(turns)),
        "approx_fees": fees,
    }


def evaluate_window(train_df, eval_df, steps, seeds, tc_list, freq):
    """Entrena el agente (varias semillas) y evalúa agente + baselines."""
    rows = []
    agent_curves = {}   # tc -> list of value arrays (una por semilla)

    # --- baselines (no dependen de entrenamiento ni de semilla del agente) ---
    L = TradingEnv.LOOKBACK
    baseline_defs = [
        ("Random",      lambda: RandomPolicy(N_ACTIONS), None),
        ("HoldCash",    lambda: HoldCash(),              None),
        ("HoldAsset0",  lambda: HoldAsset0(),            None),
        ("EqualWeight", lambda: EqualWeight(),           None),
        ("SMA",         lambda: SMA(risky_action=4, safe_action=0), L),
    ]
    baseline_curves = {}
    for tc in tc_list:
        for name, ctor, sma_L in baseline_defs:
            set_seed(0)
            vals, turns = run_policy(ctor(), eval_df, tc, sma_lookback=sma_L)
            rows.append(metrics_row(name, vals, turns, tc, freq))
            if tc == 10:
                baseline_curves[name] = vals

    # --- agente (entrenar por semilla, evaluar a cada costo) -----------------
    for seed in range(seeds):
        set_seed(seed)
        tenv = TradingEnv(train_df, transaction_cost_bps=10.0)
        obs, _ = tenv.reset()
        ag = Agent(obs_dim=obs.shape[0], n_actions=N_ACTIONS)
        ag.train(tenv, n_steps=steps)
        for tc in tc_list:
            vals, turns = run_policy(ag, eval_df, tc)
            rows.append(metrics_row(f"DQN_seed{seed}", vals, turns, tc, freq))
            agent_curves.setdefault(tc, []).append(vals)

    return rows, agent_curves, baseline_curves


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--interval", default="1h", choices=["1h", "30m", "15m"])
    ap.add_argument("--steps", type=int, default=200_000)
    ap.add_argument("--seeds", type=int, default=3)
    ap.add_argument("--tc", default="0,10,25")
    ap.add_argument("--only-last", action="store_true",
                    help="evaluar solo el último split (rápido)")
    ap.add_argument("--quick", action="store_true",
                    help="corrida corta de humo: 1 split, pocos pasos")
    args = ap.parse_args()

    if args.quick:
        args.steps, args.seeds, args.only_last = 8_000, 1, True

    tc_list = [int(x) for x in args.tc.split(",")]
    freq = FREQ[args.interval]
    os.makedirs("results", exist_ok=True)

    data = load_prices(args.interval)
    splits = SPLITS[-1:] if args.only_last else SPLITS

    all_rows = []
    for i, (train_end, eval_end) in enumerate(splits):
        train_df, eval_df = split(data, train_end, eval_end)
        wname = f"eval≤{eval_end[:4]}"
        print(f"\n=== Ventana {wname}: train {len(train_df)} | eval {len(eval_df)} ===")
        rows, agent_curves, baseline_curves = evaluate_window(
            train_df, eval_df, args.steps, args.seeds, tc_list, freq
        )
        for r in rows:
            r["window"] = wname
        all_rows.extend(rows)

        # --- figura de equity curves (a 10 bps) para esta ventana -----------
        plt.figure(figsize=(12, 5))
        idx = eval_df.index[-len(next(iter(baseline_curves.values()))):]
        for name, vals in baseline_curves.items():
            plt.plot(idx[:len(vals)], vals, lw=0.9, alpha=0.7, label=name)
        if 10 in agent_curves:
            curves = agent_curves[10]
            arr = np.array([c[:min(map(len, curves))] for c in curves])
            mean = arr.mean(0)
            plt.plot(idx[:len(mean)], mean, "k-", lw=1.6, label="DQN (media)")
            if len(curves) > 1:
                plt.fill_between(idx[:len(mean)], arr.min(0), arr.max(0),
                                 color="k", alpha=0.15, label="DQN (rango semillas)")
        plt.yscale("log")
        plt.title(f"Equity curves — {wname} (10 bps)")
        plt.ylabel("valor del portafolio")
        plt.legend(ncol=3, fontsize=8)
        plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%b-%y"))
        plt.tight_layout()
        plt.savefig(f"results/equity_{wname.replace('≤','_')}.png", dpi=130)
        plt.close()

    # --- guardar tabla de métricas -----------------------------------------
    df = pd.DataFrame(all_rows)
    df.to_csv("results/metrics.csv", index=False)

    # resumen legible: pivote de sortino y cum_ret a 10 bps
    print("\n" + "=" * 80)
    print("RESUMEN — métricas a 10 bps")
    print("=" * 80)
    sub = df[df.tc_bps == 10]
    for wname in sub.window.unique():
        print(f"\n{wname}")
        w = sub[sub.window == wname][["policy", "cum_ret", "sortino", "max_dd", "total_turnover"]]
        print(w.to_string(index=False,
              formatters={"cum_ret": "{:+.1%}".format,
                          "sortino": "{:.2f}".format,
                          "max_dd": "{:.1%}".format,
                          "total_turnover": "{:.0f}".format}))

    with open("results/summary.json", "w") as f:
        json.dump(all_rows, f, indent=2)
    print("\nResultados guardados en results/  (metrics.csv, summary.json, equity_*.png)")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile make_allocation_plot.py
"""
Allocation plot: cómo asigna capital el agente a lo largo del tiempo (eval 2024-25).
Genera results/allocation_eval_2025.png  (figura requerida por la Sección 9).
"""
import random
import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from src.data import load_prices, split
import agent as A
from agent import TradingEnv, Agent, N_ACTIONS, _ACTION_WEIGHTS, _HOLD_ACTION

random.seed(0); np.random.seed(0); torch.manual_seed(0)

data = load_prices("1h")
train_df, eval_df = split(data, "2023-12-31", "2025-12-31")

# entrenar una semilla
tenv = TradingEnv(train_df, transaction_cost_bps=10.0)
obs, _ = tenv.reset()
ag = Agent(obs_dim=obs.shape[0], n_actions=N_ACTIONS)
ag.train(tenv, n_steps=40000)

# rollout en eval, registrando los pesos efectivos en cada paso
env = TradingEnv(eval_df, transaction_cost_bps=10.0)
obs, _ = env.reset()
done = False
weights_hist, vals = [], []
while not done:
    a = ag.act(obs)
    obs, _, term, trunc, info = env.step(a)
    done = term or trunc
    weights_hist.append(env._weights.copy())
    vals.append(info["portfolio_value"])

W = np.array(weights_hist)            # (T, 4)
idx = eval_df.index[-len(W):]

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True,
                         gridspec_kw={"height_ratios": [2, 1]})

labels = ["asset_0", "asset_1", "asset_2", "cash"]
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#999999"]
axes[0].stackplot(idx, [W[:, i] for i in range(4)], labels=labels, colors=colors, alpha=0.85)
axes[0].set_title("Asignación de capital del agente en el tiempo (eval 2024-2025)")
axes[0].set_ylabel("peso del portafolio")
axes[0].set_ylim(0, 1)
axes[0].legend(loc="upper left", ncol=4, fontsize=8)

axes[1].plot(idx, vals, "k-", lw=0.9)
axes[1].axhline(10000, color="gray", ls="--", lw=0.7)
axes[1].set_title("Valor del portafolio")
axes[1].set_ylabel("valor")
axes[1].set_yscale("log")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b-%y"))

plt.tight_layout()
plt.savefig("results/allocation_eval_2025.png", dpi=130)
print("OK results/allocation_eval_2025.png")

# resumen de exposición media
print("\nExposición media del agente en eval:")
for i, lab in enumerate(labels):
    print(f"  {lab}: {W[:, i].mean():.1%}")


## 6. Corrida rápida de prueba (~1-2 min)

In [ ]:
!python run_experiment.py --quick

## 7. Último split, 3 semillas, ablación de costos (~5-15 min)

In [ ]:
!python run_experiment.py --only-last --steps 50000 --seeds 3 --tc 0,10,25

## 8. Allocation plot

In [ ]:
!python make_allocation_plot.py

## 9. Ver figuras y métricas

In [ ]:
from IPython.display import Image, display
import pandas as pd
display(Image("results/equity_eval_2025.png"))
display(Image("results/allocation_eval_2025.png"))
df = pd.read_csv("results/metrics.csv")
print("\n=== Métricas a 10 bps ==="); print(df[df.tc_bps==10][["window","policy","cum_ret","sortino","max_dd","total_turnover"]].to_string(index=False))
print("\n=== Ablación de costos (DQN_seed0) ==="); print(df[df.policy=="DQN_seed0"][["tc_bps","cum_ret","sortino","max_dd"]].to_string(index=False))

## 10. (Opcional) Walk-forward completo — las 5 ventanas

In [ ]:
!python run_experiment.py --interval 1h --steps 100000 --seeds 2 --tc 0,10,25

## 11. (Opcional) Comparar las 3 formulaciones de reward (Sección 4)
Entrena con cada reward y reporta el desempeño en el último split.

In [ ]:
import numpy as np, torch, random
from src.data import load_prices, split
from src.metrics import compute_metrics
import agent as A
from agent import Agent, N_ACTIONS
data = load_prices("1h"); train, eval_ = split(data, "2023-12-31", "2025-12-31")
def run(mode, steps=40000, seed=0):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    A.TradingEnv.REWARD_MODE = mode
    env = A.TradingEnv(train, transaction_cost_bps=10.0)
    o,_=env.reset(); ag=Agent(obs_dim=o.shape[0], n_actions=N_ACTIONS); ag.train(env, n_steps=steps)
    ee=A.TradingEnv(eval_, transaction_cost_bps=10.0); o,_=ee.reset(); done=False; vals=[]
    while not done:
        o,r,t1,t2,info=ee.step(ag.act(o)); done=t1 or t2; vals.append(info["portfolio_value"])
    m=compute_metrics(np.array(vals), freq=24*365); return m["cum_ret"], m["sortino"]
for mode in ["log","turnover","drawdown"]:
    cr,so = run(mode); print(f"{mode:>10}: cum_ret={cr:+.1%}  sortino={so:.2f}")
A.TradingEnv.REWARD_MODE = "drawdown" 

## 12. (Opcional) Experimento extra: entrenar ≤2024, evaluar 2025
Split personalizado (no toca el held-out 2026).

In [ ]:
import numpy as np, torch, random
from src.data import load_prices, split
from src.metrics import compute_metrics
import agent as A
from agent import Agent, N_ACTIONS
data = load_prices("1h"); train, eval_ = split(data, "2024-12-31", "2025-12-31")
print(f"train={len(train)} | eval 2025={len(eval_)}")
random.seed(0); np.random.seed(0); torch.manual_seed(0)
env = A.TradingEnv(train, transaction_cost_bps=10.0)
o,_=env.reset(); ag=Agent(obs_dim=o.shape[0], n_actions=N_ACTIONS); ag.train(env, n_steps=60000)
for tc in [0,10,25]:
    ee=A.TradingEnv(eval_, transaction_cost_bps=tc); o,_=ee.reset(); done=False; vals=[]
    while not done:
        o,r,t1,t2,info=ee.step(ag.act(o)); done=t1 or t2; vals.append(info["portfolio_value"])
    m=compute_metrics(np.array(vals), freq=24*365)
    print(f"  {tc:>2} bps -> cum_ret={m['cum_ret']:+.1%}  sortino={m['sortino']:.2f}  maxdd={m['max_dd']:.1%}")

## 13. (Opcional) Descargar resultados

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("/content/bellman_results", "zip", "results")
files.download("/content/bellman_results.zip")